# Student Notebook — MCP + Airbnb (Colab)

Mini-agent combining a local Notes MCP server, a stub (or real) Airbnb MCP server, and an optional real LLM planner.

In [1]:
!pip install -q mcp nest_asyncio
!pip install -q azure-ai-inference
# Uncomment if you want the real Airbnb server (needs npm):
# !npm install -g @openbnb/mcp-server-airbnb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 19.4 MB/s eta 0:00:00


In [2]:
import sys
try:
    from ipykernel.iostream import OutStream
    def _patched_fileno(self):
        return 2 if self is sys.stderr else 1
    OutStream.fileno = _patched_fileno
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2
except Exception:
    pass  # not in ipykernel, skip
print('fileno patch applied')

fileno patch applied


## Config
Flip `USE_REAL_AIRBNB` / `USE_REAL_LLM` to `True` when you have npm / GitHub token.
Leave as `False` for a fully stubbed run (no keys needed).

In [3]:
import os
from pathlib import Path

MCP_HTTP_TOKEN  = os.getenv('MCP_HTTP_TOKEN', 'devtoken123')
USE_REAL_AIRBNB = False   # True → needs npm + @openbnb/mcp-server-airbnb
USE_REAL_LLM    = False   # True → needs GITHUB_TOKEN in Colab secrets

BASE_ENV = os.environ.copy()
BASE_ENV['MCP_HTTP_TOKEN'] = MCP_HTTP_TOKEN
print(f'USE_REAL_AIRBNB={USE_REAL_AIRBNB}  USE_REAL_LLM={USE_REAL_LLM}')

USE_REAL_AIRBNB=False  USE_REAL_LLM=False


## GitHub Token (optional)
Only needed when `USE_REAL_LLM = True`. Safe to skip otherwise.

In [4]:
import os
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('GITHUB_TOKEN loaded from Colab secrets.')
except Exception:
    os.environ.setdefault('GITHUB_TOKEN', '')
    print('No GITHUB_TOKEN — stub LLM will be used.')

No GITHUB_TOKEN — stub LLM will be used.


## Write MCP server files to disk

In [5]:
from pathlib import Path

# ── Notes server ──────────────────────────────────────────────
LOCAL_SERVER = Path('local_notes_server.py')
LOCAL_SERVER.write_text('''
from mcp.server.fastmcp import FastMCP
notes = []
mcp = FastMCP("notes")

@mcp.tool()
def add_note(text: str) -> str:
    "Add a note to the in-memory list."
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    "List saved notes."
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
'''.strip(), encoding='utf-8')

# ── Stub Airbnb server ─────────────────────────────────────────
STUB_AIRBNB_SERVER = Path('stub_airbnb_server.py')
STUB_AIRBNB_SERVER.write_text('''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("airbnb-stub")

@mcp.tool()
def airbnb_search(location: str, checkin: str = "", checkout: str = "", adults: int = 2) -> str:
    "Search Airbnb listings (stub)."
    return (
        f"[STUB] Listings in {location} ({checkin} to {checkout}, {adults} adults):\\n"
        "1. Cozy Studio  - $85/night  4.7 stars\\n"
        "2. Modern Flat  - $120/night 4.9 stars\\n"
        "3. Sunny Loft   - $95/night  4.8 stars\\n"
    )

@mcp.tool()
def airbnb_listing_details(listing_id: str) -> str:
    "Get details for a listing (stub)."
    return f"[STUB] Listing {listing_id}: 2 beds, WiFi, kitchen, free cancellation."

if __name__ == "__main__":
    mcp.run()
'''.strip(), encoding='utf-8')

print('Wrote:', LOCAL_SERVER, 'and', STUB_AIRBNB_SERVER)

Wrote: local_notes_server.py and stub_airbnb_server.py


## Client helpers

In [6]:
import asyncio, json, nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


# ── Tool schema converter ──────────────────────────────────────
def convert_tool(tool, prefix: str) -> Dict:
    return {
        'type': 'function',
        'function': {
            'name': f'{prefix}__{tool.name}',
            'description': tool.description or 'mcp tool',
            'parameters': {
                'type': 'object',
                'properties': tool.inputSchema.get('properties', {}),
                'required':   tool.inputSchema.get('required', []),
            },
        },
    }


# ── Stub planner (no LLM needed) ──────────────────────────────
def stub_planner(prompt: str, functions: List[Dict]) -> List[Dict]:
    calls = []
    for f in functions:
        name = f['function']['name']
        if name.endswith('airbnb_search'):
            calls.append({'name': name, 'args': {
                'location': 'Paris', 'checkin': '2025-08-01',
                'checkout': '2025-08-07', 'adults': 2}})
        elif name.endswith('add_note'):
            calls.append({'name': name, 'args': {'text': 'Paris trip planned for August 2025'}})
        elif name.endswith('list_notes'):
            calls.append({'name': name, 'args': {}})
    return calls


# ── LLM planner (real, needs GITHUB_TOKEN) ────────────────────
def call_llm(prompt: str, functions: List[Dict], use_real: bool = False) -> List[Dict]:
    if not use_real:
        print('[stub planner] returning hardcoded tool_calls')
        return stub_planner(prompt, functions)

    from azure.ai.inference import ChatCompletionsClient
    from azure.ai.inference.models import SystemMessage, UserMessage
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('Set GITHUB_TOKEN or use USE_REAL_LLM=False')

    client = ChatCompletionsClient(
        'https://models.inference.ai.azure.com', AzureKeyCredential(token))
    resp = client.complete(
        model='gpt-4o',
        messages=[
            SystemMessage('You are a planner. Call the right tools for the user request.'),
            UserMessage(prompt),
        ],
        tools=functions,
        tool_choice='auto',
        temperature=0,
        max_tokens=512,
    )
    calls = []
    for tc in resp.choices[0].message.tool_calls or []:
        raw = tc.function.arguments
        calls.append({'name': tc.function.name,
                      'args': json.loads(raw) if isinstance(raw, str) else raw})
    return calls

print('helpers defined')

helpers defined


## Orchestrate

In [7]:
async def orchestrate(prompt: str):
    local_params = StdioServerParameters(
        command='mcp', args=['run', str(LOCAL_SERVER)], env=BASE_ENV)

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command='npx',
            args=['@openbnb/mcp-server-airbnb', '--ignore-robots-txt'],
            env=BASE_ENV)
    else:
        airbnb_params = StdioServerParameters(
            command='mcp', args=['run', str(STUB_AIRBNB_SERVER)], env=BASE_ENV)

    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = (await local_sess.list_tools()).tools

            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = (await airbnb_sess.list_tools()).tools

                    functions = (
                        [convert_tool(t, 'notes')  for t in local_tools] +
                        [convert_tool(t, 'airbnb') for t in airbnb_tools]
                    )
                    print('Available tools:', [f['function']['name'] for f in functions])

                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print('Planned tool_calls:', tool_calls)

                    tool_results = []
                    for call in tool_calls:
                        name, args = call['name'], call['args']
                        prefix, tool_name = name.split('__', 1)
                        if prefix == 'notes':
                            res = await local_sess.call_tool(tool_name, args)
                        else:
                            res = await airbnb_sess.call_tool(tool_name, args)
                        tool_results.append({
                            'name': name, 'args': args,
                            'content': [c.text for c in res.content if hasattr(c, 'text')],
                        })

                    return tool_calls, tool_results

print('orchestrate defined')

orchestrate defined


## Demo
Run end-to-end. Adjust the prompt freely.

In [8]:
import traceback

prompt = (
    'Search for Airbnb listings in Paris for 2 adults '
    'from 2025-08-01 to 2025-08-07. '
    "Then save a note: 'Paris trip planned for August 2025'. "
    'Finally, list all saved notes.'
)

# Unwrap ExceptionGroup so the real error is visible
async def safe_orchestrate(p):
    try:
        return await orchestrate(p)
    except* Exception as eg:
        for exc in eg.exceptions:
            traceback.print_exception(type(exc), exc, exc.__traceback__)
        raise

tool_calls, tool_results = asyncio.run(safe_orchestrate(prompt))

print('\n' + '='*60)
print('TOOL CALLS')
print('='*60)
for tc in tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

print('\n' + '='*60)
print('TOOL RESULTS')
print('='*60)
for res in tool_results:
    print(f"\n[{res['name']}]")
    for chunk in res['content']:
        print(str(chunk)[:2000])

if USE_REAL_LLM:
    print('\n' + '='*60)
    print('LLM SYNTHESIS')
    print('='*60)
    print(answer_with_llm(prompt, tool_calls, tool_results))

Available tools: ['notes__add_note', 'notes__list_notes', 'airbnb__airbnb_search', 'airbnb__airbnb_listing_details']
[stub planner] returning hardcoded tool_calls
Planned tool_calls: [{'name': 'notes__add_note', 'args': {'text': 'Paris trip planned for August 2025'}}, {'name': 'notes__list_notes', 'args': {}}, {'name': 'airbnb__airbnb_search', 'args': {'location': 'Paris', 'checkin': '2025-08-01', 'checkout': '2025-08-07', 'adults': 2}}]

TOOL CALLS
  → notes__add_note({'text': 'Paris trip planned for August 2025'})
  → notes__list_notes({})
  → airbnb__airbnb_search({'location': 'Paris', 'checkin': '2025-08-01', 'checkout': '2025-08-07', 'adults': 2})

TOOL RESULTS

[notes__add_note]
Saved note #1: Paris trip planned for August 2025

[notes__list_notes]
1. Paris trip planned for August 2025

[airbnb__airbnb_search]
[STUB] Listings in Paris (2025-08-01 to 2025-08-07, 2 adults):
1. Cozy Studio  - $85/night  4.7 stars
2. Modern Flat  - $120/night 4.9 stars
3. Sunny Loft   - $95/night  4.